# Importância de Features - Melhor Modelo Tabular

Este notebook treina o melhor modelo tabular, extrai a importância nativa das features e calcula a importância por permutação.

Modelo analisado: `RandomForest sem gender`, escolhido como melhor challenger operacional após a rodada de ablação.

## 1. Importação de Bibliotecas

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from tech_challenge_churn.config import RANDOM_SEED
from tech_challenge_churn.data.load import read_raw_data, split_features_target
from tech_challenge_churn.data.schema import validate_telco_schema
from tech_challenge_churn.models.feature_ablation import (
    build_ablation_model,
    build_ablation_registry,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

## 2. Carregamento do Dataset

Carregamos a base original, validamos o schema e separamos `Churn` como alvo binário. O pipeline do modelo fará a engenharia de features, imputação, escala e one-hot internamente.

In [ ]:
data_path = PROJECT_ROOT / "Telco-Customer-Churn.csv"
raw_data = validate_telco_schema(read_raw_data(data_path))
features, target = split_features_target(raw_data)

print(f"Dataset bruto: {raw_data.shape}")
print(f"Features de entrada: {features.shape}")
print(f"Taxa de churn: {target.mean():.2%}")
display(features.head())

## 3. Treinamento do Melhor Modelo

A rodada de ablação indicou que remover `gender` preserva e melhora levemente o F1. Por isso, o modelo analisado aqui usa o feature set `no_gender`.

In [ ]:
best_spec = next(spec for spec in build_ablation_registry() if spec.name == "no_gender")
model = build_ablation_model(best_spec)

x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    stratify=target,
    random_state=RANDOM_SEED,
)

model.fit(x_train, y_train)
y_proba = model.predict_proba(x_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

metrics = {
    "roc_auc": roc_auc_score(y_test, y_proba),
    "pr_auc": average_precision_score(y_test, y_proba),
    "f1_threshold_0_5": f1_score(y_test, y_pred),
}

display(pd.Series(metrics, name="holdout_metrics").to_frame())

## 4. Importância Nativa do RandomForest

O RandomForest expõe `feature_importances_`, calculado a partir da redução média de impureza das árvores. Essa importância é extraída depois do pré-processamento, portanto considera as colunas finais geradas por escala, engenharia de features e one-hot.

In [ ]:
feature_pipeline = model.named_steps["features"]
preprocessor = feature_pipeline.named_steps["preprocessor"]
classifier = model.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
native_importance = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": classifier.feature_importances_,
    }
).sort_values("importance", ascending=False)

print(f"Features finais analisadas: {len(native_importance)}")
display(native_importance.head(30))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
sns.barplot(
    data=native_importance.head(25),
    x="importance",
    y="feature",
    color="#1f77b4",
    ax=ax,
)
ax.set_title("Importância Nativa do RandomForest - Top 25")
ax.set_xlabel("Importância por redução de impureza")
ax.set_ylabel("Feature final")
plt.tight_layout()

### Agregação por Feature de Origem

Como o one-hot transforma uma variável categórica em várias colunas, agregamos as importâncias das colunas finais para facilitar a leitura por variável de negócio.

In [ ]:
numeric_features = list(preprocessor.transformers_[0][2])
categorical_features = list(preprocessor.transformers_[1][2])

def feature_group(feature_name: str) -> str:
    """Mapeia uma coluna final do pipeline para a feature de origem."""
    clean_name = feature_name.replace("numeric__", "").replace("categorical__", "")
    for categorical_feature in sorted(categorical_features, key=len, reverse=True):
        if clean_name == categorical_feature or clean_name.startswith(f"{categorical_feature}_"):
            return categorical_feature
    for numeric_feature in numeric_features:
        if clean_name == numeric_feature:
            return numeric_feature
    return clean_name

grouped_native_importance = (
    native_importance.assign(feature_group=native_importance["feature"].map(feature_group))
    .groupby("feature_group", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

display(grouped_native_importance.head(30))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    data=grouped_native_importance.head(20),
    x="importance",
    y="feature_group",
    color="#2ca02c",
    ax=ax,
)
ax.set_title("Importância Nativa Agregada por Feature de Origem - Top 20")
ax.set_xlabel("Importância agregada")
ax.set_ylabel("Feature de origem")
plt.tight_layout()

## 5. Importância por Permutação

A importância por permutação mede a queda de desempenho quando embaralhamos uma coluna no holdout. Aqui a permutação é feita nas colunas originais de entrada do pipeline. Como o feature set recomendado remove `gender`, essa coluna deve ter importância próxima de zero.

In [ ]:
permutation = permutation_importance(
    model,
    x_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

permutation_importance_df = pd.DataFrame(
    {
        "feature": x_test.columns,
        "importance_mean": permutation.importances_mean,
        "importance_std": permutation.importances_std,
    }
).sort_values("importance_mean", ascending=False)

display(permutation_importance_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top_permutation = permutation_importance_df.head(20).reset_index(drop=True)

sns.barplot(
    data=top_permutation,
    x="importance_mean",
    y="feature",
    color="#ff7f0e",
    ax=ax,
)
ax.errorbar(
    top_permutation["importance_mean"],
    range(len(top_permutation)),
    xerr=top_permutation["importance_std"],
    fmt="none",
    c="black",
    capsize=3,
)
ax.set_title("Importância por Permutação no Holdout - F1")
ax.set_xlabel("Queda média de F1 ao permutar a coluna")
ax.set_ylabel("Feature original")
plt.tight_layout()

## 6. Comparação entre Importância Nativa e Permutação

- A importância nativa olha para as features finais do modelo após engenharia e one-hot.
- A importância por permutação olha para as colunas originais de entrada e mede impacto no F1 do holdout.
- Quando uma variável é usada apenas para criar features derivadas, a importância pode aparecer mais forte na visão nativa do que na permutação da coluna original.
- Se uma coluna for ignorada pelo pipeline, como `gender` no feature set `no_gender`, a importância por permutação deve ficar próxima de zero.

In [ ]:
output_dir = PROJECT_ROOT / "reports" / "feature_importance"
output_dir.mkdir(parents=True, exist_ok=True)

native_importance.to_csv(output_dir / "random_forest_no_gender_native_importance.csv", index=False)
grouped_native_importance.to_csv(
    output_dir / "random_forest_no_gender_native_importance_grouped.csv",
    index=False,
)
permutation_importance_df.to_csv(
    output_dir / "random_forest_no_gender_permutation_importance.csv",
    index=False,
)

print(f"Artefatos salvos em: {output_dir}")

## 7. Registro no MLflow

A célula abaixo registra as tabelas de importância como artefatos do experimento `telco-churn-feature-importance`.

In [ ]:
mlflow.set_tracking_uri((PROJECT_ROOT / "mlruns").as_uri())
mlflow.set_experiment("telco-churn-feature-importance")

with mlflow.start_run(run_name="random_forest_no_gender_feature_importance"):
    mlflow.set_tags(
        {
            "stage": "interpretability",
            "model": "random_forest_no_gender",
            "feature_set": best_spec.name,
        }
    )
    mlflow.log_params(
        {
            "native_importance": "random_forest_feature_importances",
            "permutation_scoring": "f1",
            "permutation_repeats": 10,
            "final_feature_count": len(native_importance),
            "input_feature_count": x_test.shape[1],
        }
    )
    mlflow.log_metrics(metrics)
    for artifact_path in output_dir.glob("random_forest_no_gender_*.csv"):
        mlflow.log_artifact(str(artifact_path), artifact_path="feature_importance")

print("Importâncias registradas no MLflow.")

## 8. Leitura Esperada

As variáveis ligadas a contrato, tempo de relacionamento, cobrança mensal, cobrança total, método de pagamento e serviços de suporte/proteção tendem a concentrar a maior parte do sinal de churn. Essa leitura está alinhada com a EDA, a ablação e a recomendação final do projeto.